In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.integrate import odeint
from scipy.optimize import minimize
import os

# ==========================================
# 1. User settings and physical properties
# ==========================================
N_grid = 40  # 40 is recommended for fitting speed; 100-200 can be used for higher resolution

# Input file path
file_path = 'data/260204_numerical_solution.xlsx'

# Material properties
L_aei, iec_aei, rho_aei, wu_aei = 0.896e-4, 1.7, 1.12, 50.4
L_cei, iec_cei, rho_cei, wu_cei = 0.51e-4, 0.91, 1.95, 19.4
C_bulk = 0.1
dp = 0.45e-4

# AEI 3-ion model settings
C_bulk_NO3 = 0.1
C_bulk_K = 0.1
C_bulk_Cl = 0.0

# ==========================================
# 2. Physics engine
# ==========================================
def calculate_c_fix(iec, rho_dry, wu):
    vol_wet = (1.0 / rho_dry) + (wu / 100.0)
    return (iec * 1e-3) / (vol_wet * 1e-3)


c_fix_aei = calculate_c_fix(iec_aei, rho_aei, wu_aei)
c_fix_cei = calculate_c_fix(iec_cei, rho_cei, wu_cei)


def get_donnan_boundary_3ion(c_fix, c_b_no3, c_b_cl, c_b_k):
    A = c_b_no3 + c_b_cl + 1e-15
    B = -c_fix
    C = -c_b_k
    k = (-B + np.sqrt(B**2 - 4 * A * C)) / (2 * A)
    return c_b_no3 * k, c_b_cl * k, c_b_k / k


# AEI 3-ion solver
def pde_solver_aei_3ion(t_eval, L, c_fix, D_vals, full_output=False):
    D_NO3, D_Cl, D_K = D_vals
    N = N_grid
    x = np.linspace(0, L, N)
    dx = x[1] - x[0]

    c_s_NO3, c_s_Cl, c_s_K = get_donnan_boundary_3ion(c_fix, C_bulk_NO3, C_bulk_Cl, C_bulk_K)
    z_no3, z_cl, z_k = -1, -1, 1

    def model(y, t):
        c_no3 = y[:N]
        c_k = y[N:]
        c_cl = np.maximum(c_k + c_fix - c_no3, 1e-15)

        dcdt_no3, dcdt_k = np.zeros(N), np.zeros(N)
        J_no3, J_k = np.zeros(N + 1), np.zeros(N + 1)

        for i in range(N - 1):
            m_no3 = 0.5 * (c_no3[i] + c_no3[i + 1])
            m_k = 0.5 * (c_k[i] + c_k[i + 1])
            m_cl = 0.5 * (c_cl[i] + c_cl[i + 1])
            g_no3 = (c_no3[i + 1] - c_no3[i]) / dx
            g_k = (c_k[i + 1] - c_k[i]) / dx
            g_cl = (c_cl[i + 1] - c_cl[i]) / dx

            num = (z_no3 * D_NO3 * g_no3) + (z_cl * D_Cl * g_cl) + (z_k * D_K * g_k)
            den = (z_no3**2 * D_NO3 * m_no3) + (z_cl**2 * D_Cl * m_cl) + (z_k**2 * D_K * m_k) + 1e-15
            Psi = -num / den

            J_no3[i + 1] = -D_NO3 * (g_no3 + z_no3 * m_no3 * Psi)
            J_k[i + 1] = -D_K * (g_k + z_k * m_k * Psi)

        g_no3_0 = (c_no3[0] - c_s_NO3) / (dx / 2)
        g_k_0 = (c_k[0] - c_s_K) / (dx / 2)
        g_cl_0 = (c_cl[0] - c_s_Cl) / (dx / 2)
        m_no3_0 = 0.5 * (c_no3[0] + c_s_NO3)
        m_k_0 = 0.5 * (c_k[0] + c_s_K)
        m_cl_0 = 0.5 * (c_cl[0] + c_s_Cl)

        num_0 = (z_no3 * D_NO3 * g_no3_0) + (z_cl * D_Cl * g_cl_0) + (z_k * D_K * g_k_0)
        den_0 = (z_no3**2 * D_NO3 * m_no3_0) + (z_cl**2 * D_Cl * m_cl_0) + (z_k**2 * D_K * m_k_0) + 1e-15
        Psi_0 = -num_0 / den_0

        J_no3[0] = -D_NO3 * (g_no3_0 + z_no3 * m_no3_0 * Psi_0)
        J_k[0] = -D_K * (g_k_0 + z_k * m_k_0 * Psi_0)
        J_no3[N], J_k[N] = 0, 0

        for i in range(N):
            dcdt_no3[i] = -(J_no3[i + 1] - J_no3[i]) / dx
            dcdt_k[i] = -(J_k[i + 1] - J_k[i]) / dx

        return np.concatenate((dcdt_no3, dcdt_k))

    t_solve = np.concatenate(([0], t_eval)) if t_eval[0] != 0 else t_eval
    y0 = np.concatenate((np.zeros(N) + 1e-15, np.zeros(N) + 1e-15))
    sol = odeint(model, y0, t_solve, mxstep=5000)

    sol_no3 = sol[:, :N]
    sol_k = sol[:, N:]
    sol_cl = np.maximum(sol_k + c_fix - sol_no3, 1e-15)

    if full_output:
        return {'time': t_solve, 'x': x, 'conc_no3': sol_no3, 'conc_k': sol_k, 'conc_cl': sol_cl}

    dist = L - x
    evanescent = np.exp(-2 * dist / dp)
    signals = np.array([np.trapz(c, x=x) for c in sol_no3 * evanescent])
    return signals[1:] if t_eval[0] != 0 else signals


# CEI 2-ion solver
def pde_solver_cei_2ion(t_eval, L, c_fix, D_in, D_out, full_output=False):
    N = N_grid
    x = np.linspace(0, L, N)
    dx = x[1] - x[0]
    c_source = (-c_fix + np.sqrt(c_fix**2 + 4 * C_bulk**2)) / 2

    def model(c, t):
        num = D_in * D_out * (2 * c + c_fix)
        den = D_in * c + D_out * (c + c_fix) + 1e-15
        D_eff = num / den

        J = np.zeros(N + 1)
        for i in range(N - 1):
            D_mid = 0.5 * (D_eff[i] + D_eff[i + 1])
            J[i + 1] = -D_mid * (c[i + 1] - c[i]) / dx
        J[0] = -D_eff[0] * (c[0] - c_source) / (dx / 2)
        J[N] = 0

        dcdt = np.zeros(N)
        for i in range(N):
            dcdt[i] = -(J[i + 1] - J[i]) / dx
        return dcdt

    t_solve = np.concatenate(([0], t_eval)) if t_eval[0] != 0 else t_eval
    sol = odeint(model, np.zeros(N), t_solve, mxstep=5000)

    if full_output:
        return {'time': t_solve, 'x': x, 'conc': sol}

    dist = L - x
    evanescent = np.exp(-2 * dist / dp)
    signals = np.array([np.trapz(c, x=x) for c in sol * evanescent])
    return signals[1:] if t_eval[0] != 0 else signals


# ==========================================
# 3. Load data and run fitting
# ==========================================
try:
    df = pd.read_excel(file_path, engine='openpyxl')
except Exception:
    print('Input file could not be loaded. Using dummy data instead.')
    time_exp = np.linspace(0, 100, 20)
    sig_aei_exp = 1 - np.exp(-time_exp / 20)
    sig_cei_exp = 1 - np.exp(-time_exp / 10)
else:
    time_exp = df['time (s)'].values
    sig_aei_exp = df['PSIm area'].values
    sig_cei_exp = df['Nafion area'].values

print(f'Starting optimization with grid size = {N_grid}...')


def objective_aei(params):
    D_NO3, D_Cl, D_K = 10**params[0], 10**params[1], 10**params[2]
    if np.any(np.array([D_NO3, D_Cl, D_K]) > 1e-8):
        return 1e6
    if np.any(np.array([D_NO3, D_Cl, D_K]) < 1e-15):
        return 1e6
    if D_NO3 > 3.0 * D_Cl or D_Cl > 3.0 * D_NO3:
        return 1e6
    if D_K >= D_NO3 or D_K >= D_Cl:
        return 1e6
    try:
        sim = pde_solver_aei_3ion(time_exp, L_aei, c_fix_aei, [D_NO3, D_Cl, D_K])
    except Exception:
        return 1e6
    if np.max(sim) == 0:
        return 1e6
    return np.mean((sim / np.max(sim) - sig_aei_exp / np.max(sig_aei_exp))**2)


res_aei = minimize(objective_aei, [np.log10(1e-11), np.log10(2e-11), np.log10(1e-12)], method='Nelder-Mead')
best_aei = 10**res_aei.x


def objective_cei(params):
    D_NO3, D_K = 10**params[0], 10**params[1]
    if D_K < 100 * D_NO3:
        return 1e6
    sim = pde_solver_cei_2ion(time_exp, L_cei, c_fix_cei, D_NO3, D_K)
    if np.max(sim) == 0:
        return 1e6
    return np.mean((sim / np.max(sim) - sig_cei_exp / np.max(sig_cei_exp))**2)


res_cei = minimize(objective_cei, [np.log10(1e-11), np.log10(5e-9)], method='Nelder-Mead')
best_cei = 10**res_cei.x

print(f'[AEI result] NO3: {best_aei[0]:.2e}, Cl: {best_aei[1]:.2e}, K: {best_aei[2]:.2e}')
print(f'[CEI result] NO3: {best_cei[0]:.2e}, K: {best_cei[1]:.2e}')


# ==========================================
# 4. Final analysis and visualization
# ==========================================
print(f'Generating simulation outputs (N={N_grid})...')

T_FINAL = max(time_exp)
t_full = np.linspace(0, T_FINAL, 200)

sim_sig_aei = pde_solver_aei_3ion(t_full, L_aei, c_fix_aei, best_aei)
sim_sig_cei = pde_solver_cei_2ion(t_full, L_cei, c_fix_cei, best_cei[0], best_cei[1])
scale_a = np.max(sig_aei_exp) / np.max(sim_sig_aei) if np.max(sim_sig_aei) > 0 else 1
scale_c = np.max(sig_cei_exp) / np.max(sim_sig_cei) if np.max(sim_sig_cei) > 0 else 1

res_aei_full = pde_solver_aei_3ion(t_full, L_aei, c_fix_aei, best_aei, full_output=True)
res_cei_full = pde_solver_cei_2ion(t_full, L_cei, c_fix_cei, best_cei[0], best_cei[1], full_output=True)
conc_cei_k_full = res_cei_full['conc'] + c_fix_cei

fig = plt.figure(figsize=(15, 12))
gs = fig.add_gridspec(3, 2, hspace=0.4, wspace=0.3)

ax1 = fig.add_subplot(gs[0, 0])
ax1.scatter(time_exp, sig_aei_exp, c='r', alpha=0.5, label='Exp: AEI')
ax1.plot(t_full, sim_sig_aei * scale_a, 'r--', label='Sim: AEI')
ax1.scatter(time_exp, sig_cei_exp, c='b', alpha=0.5, label='Exp: CEI')
ax1.plot(t_full, sim_sig_cei * scale_c, 'b--', label='Sim: CEI')
ax1.set_title(f'Signal fitting (N={N_grid})')
ax1.set_xlabel('Time (s)')
ax1.legend()
ax1.grid(True)

ax2 = fig.add_subplot(gs[0, 1])
ax2.axis('off')
ax2.text(
    0.5,
    0.5,
    'Best-fit diffusion coefficients

'
    + f'AEI NO3: {best_aei[0]:.2e} m$^2$/s
'
    + f'AEI Cl: {best_aei[1]:.2e} m$^2$/s
'
    + f'AEI K: {best_aei[2]:.2e} m$^2$/s

'
    + f'CEI NO3: {best_cei[0]:.2e} m$^2$/s
'
    + f'CEI K: {best_cei[1]:.2e} m$^2$/s',
    ha='center',
    va='center',
    fontsize=12,
)


def plot_heatmap(ax, data, title, cmap, L_um, time_arr):
    sns.heatmap(data.T, ax=ax, cmap=cmap, cbar_kws={'label': 'M'})
    xticks = np.linspace(0, len(time_arr) - 1, 5, dtype=int)
    xticklabels = [f'{time_arr[i]:.0f}' for i in xticks]
    ax.set_xticks(xticks)
    ax.set_xticklabels(xticklabels, rotation=0)
    ax.set_xlabel('Time (s)')

    yticks = np.linspace(0, data.shape[1] - 1, 5, dtype=int)
    yticklabels = [f'{val:.1f}' for val in np.linspace(0, L_um, 5)]
    ax.set_yticks(yticks)
    ax.set_yticklabels(yticklabels)
    ax.set_ylabel('Depth (µm)')
    ax.invert_yaxis()
    ax.set_title(title)


ax3 = fig.add_subplot(gs[1, 0])
plot_heatmap(ax3, res_aei_full['conc_no3'], '[AEI] NO3', 'Reds', L_aei * 1e6, t_full)

ax4 = fig.add_subplot(gs[1, 1])
plot_heatmap(ax4, res_aei_full['conc_cl'], '[AEI] Cl', 'Greens', L_aei * 1e6, t_full)

ax5 = fig.add_subplot(gs[2, 0])
plot_heatmap(ax5, res_aei_full['conc_k'], '[AEI] K', 'Oranges', L_aei * 1e6, t_full)

ax6 = fig.add_subplot(gs[2, 1])
plot_heatmap(ax6, res_cei_full['conc'], '[CEI] NO3', 'Blues', L_cei * 1e6, t_full)

plt.tight_layout()


# ==========================================
# 5. Export data
# ==========================================
save_dir = os.path.dirname(file_path)
file_name_only = os.path.splitext(os.path.basename(file_path))[0]
save_path = os.path.join(save_dir, f'{file_name_only}_simulation_result_cleaned.xlsx')

print(f'Exporting simulation data to {save_path}')

df_signal = pd.DataFrame({
    'Time (s)': t_full,
    'AEI_Sim_Signal': sim_sig_aei * scale_a,
    'CEI_Sim_Signal': sim_sig_cei * scale_c,
})

df_aei_no3_map = pd.DataFrame(res_aei_full['conc_no3'], index=t_full, columns=res_aei_full['x'])
df_aei_cl_map = pd.DataFrame(res_aei_full['conc_cl'], index=t_full, columns=res_aei_full['x'])
df_aei_k_map = pd.DataFrame(res_aei_full['conc_k'], index=t_full, columns=res_aei_full['x'])
df_cei_no3_map = pd.DataFrame(res_cei_full['conc'], index=t_full, columns=res_cei_full['x'])
df_cei_k_map = pd.DataFrame(conc_cei_k_full, index=t_full, columns=res_cei_full['x'])

with pd.ExcelWriter(save_path, engine='openpyxl') as writer:
    df_signal.to_excel(writer, sheet_name='Signal_Fitting', index=False)
    df_aei_no3_map.to_excel(writer, sheet_name='AEI_NO3_Map')
    df_aei_cl_map.to_excel(writer, sheet_name='AEI_Cl_Map')
    df_aei_k_map.to_excel(writer, sheet_name='AEI_K_Map')
    df_cei_no3_map.to_excel(writer, sheet_name='CEI_NO3_Map')
    df_cei_k_map.to_excel(writer, sheet_name='CEI_K_Map')

print('Export completed.')
plt.show()
